# Option Greeks in one adjoint pass

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/auto-differentiation/xad-py/blob/main/notebooks/02_option_greeks.ipynb)

A pricer maps market inputs (spot, strike, rate, volatility, maturity) to a price. Risk management needs the
sensitivities of that price, the *Greeks*. Bumping each input costs one or two extra pricings per Greek. With XAD's
adjoint mode, **all of them come out of a single backward pass**.

We do this twice:

1. The closed-form Black-Scholes formula, checked against the textbook Greek formulas.
2. A Monte Carlo pricer, where bumping is slow and noisy and AD gives pathwise Greeks directly.

In [1]:
%pip install -q xad

Note: you may need to restart the kernel to use updated packages.


In [2]:
import math
import random
import time

import xad.adj_1st as xadj
from xad import math as xm

# market data and contract
inputs = {"spot": 100.0, "strike": 105.0, "rate": 0.03, "vol": 0.20, "maturity": 1.0}

A small helper: register the inputs, record the pricer, run one backward pass, and return the price and the
derivative with respect to each input.

In [3]:
def price_and_greeks(pricer, inputs):
    names = list(inputs)
    args = [xadj.Real(inputs[n]) for n in names]
    with xadj.Tape() as tape:
        tape.registerInputs(args)
        tape.newRecording()
        price = pricer(*args)
        tape.registerOutput(price)
        price.derivative = 1.0
        tape.computeAdjoints()
        return price.value, {n: a.derivative for n, a in zip(names, args)}

## 1. Black-Scholes, closed form

A European call. `xm.erf` gives the normal CDF, so the formula works for floats and XAD numbers alike.

In [4]:
def norm_cdf(x):
    return 0.5 * (1.0 + xm.erf(x / math.sqrt(2.0)))


def black_scholes_call(spot, strike, rate, vol, maturity):
    sqrt_t = xm.sqrt(maturity)
    d1 = (xm.log(spot / strike) + (rate + 0.5 * vol * vol) * maturity) / (vol * sqrt_t)
    d2 = d1 - vol * sqrt_t
    return spot * norm_cdf(d1) - strike * xm.exp(-rate * maturity) * norm_cdf(d2)


bs_price, bs_greeks = price_and_greeks(black_scholes_call, inputs)
print(f"price = {bs_price:.6f}")

price = 7.128065


Compare with the analytic Greeks. Delta, vega and rho are the usual ones; the maturity sensitivity is minus
theta; the strike sensitivity is the (discounted) digital price.

In [5]:
S, K, r, v, T = inputs.values()
d1 = (math.log(S / K) + (r + 0.5 * v * v) * T) / (v * math.sqrt(T))
d2 = d1 - v * math.sqrt(T)
pdf = math.exp(-0.5 * d1 * d1) / math.sqrt(2 * math.pi)
N = lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2)))
analytic = {
    "spot": N(d1),                                                    # delta
    "strike": -math.exp(-r * T) * N(d2),
    "rate": K * T * math.exp(-r * T) * N(d2),                         # rho
    "vol": S * pdf * math.sqrt(T),                                    # vega
    "maturity": S * pdf * v / (2 * math.sqrt(T)) + r * K * math.exp(-r * T) * N(d2),  # -theta
}

print(f"{'d price / d':12} {'XAD adjoint':>14} {'analytic':>14} {'abs diff':>10}")
for n in inputs:
    print(f"{n:12} {bs_greeks[n]:14.10f} {analytic[n]:14.10f} {abs(bs_greeks[n] - analytic[n]):10.1e}")

d price / d     XAD adjoint       analytic   abs diff
spot           0.5024132586   0.5024132586    3.3e-16
strike        -0.4106024875  -0.4106024875    3.3e-16
rate          43.1132611915  43.1132611915    3.6e-14
vol           39.8934981307  39.8934981307    0.0e+00
maturity       5.2827476488   5.2827476488    8.9e-16


Exact to machine precision, from code that only describes the price.

## 2. Monte Carlo

The same option priced by simulating terminal spot prices under geometric Brownian motion. With bumping, each Greek
needs extra simulations with the same random numbers, and the bump size trades bias against noise. Adjoint mode
differentiates every path exactly (pathwise Greeks), so all five come from one simulation and one backward pass.

In [6]:
N_PATHS = 50_000
rng = random.Random(42)
normals = [rng.gauss(0.0, 1.0) for _ in range(N_PATHS)]


def mc_call(spot, strike, rate, vol, maturity):
    drift = (rate - 0.5 * vol * vol) * maturity
    diffusion = vol * xm.sqrt(maturity)
    payoff_sum = 0.0
    for z in normals:
        s_t = spot * xm.exp(drift + diffusion * z)
        payoff_sum = payoff_sum + xm.max(s_t - strike, 0.0)
    return xm.exp(-rate * maturity) * payoff_sum / len(normals)


t0 = time.perf_counter()
mc_price, mc_greeks = price_and_greeks(mc_call, inputs)
t_adj = time.perf_counter() - t0

print(f"MC price = {mc_price:.4f}   (closed form {bs_price:.4f})")
print(f"price + 5 Greeks in {t_adj:.2f} s\n")
print(f"{'d price / d':12} {'MC adjoint':>12} {'analytic':>12}")
for n in inputs:
    print(f"{n:12} {mc_greeks[n]:12.4f} {analytic[n]:12.4f}")

MC price = 7.1448   (closed form 7.1281)
price + 5 Greeks in 0.79 s

d price / d    MC adjoint     analytic
spot               0.5036       0.5024
strike            -0.4115      -0.4106
rate              43.2103      43.1133
vol               39.9760      39.8935
maturity           5.2939       5.2827


The Monte Carlo Greeks match the closed form within the simulation error of 50,000 paths.

For comparison, the plain-float pricer and what central-difference bumping of all five inputs would cost:

In [7]:
t0 = time.perf_counter()
mc_call(*inputs.values())
t_plain = time.perf_counter() - t0

print(f"one plain pricing:              {t_plain:.2f} s")
print(f"bumping, 5 inputs x 2 pricings: {10 * t_plain:.2f} s (estimate)")
print(f"XAD adjoint, price + 5 Greeks:  {t_adj:.2f} s")

one plain pricing:              0.47 s
bumping, 5 inputs x 2 pricings: 4.67 s (estimate)
XAD adjoint, price + 5 Greeks:  0.79 s


The advantage grows with the number of inputs: a real book depends on full yield curves and volatility surfaces
with hundreds of points, and the adjoint cost stays about the same.

## Next steps

- [Adjoint vs finite differences at scale](https://colab.research.google.com/github/auto-differentiation/xad-py/blob/main/notebooks/03_adjoint_vs_finite_differences.ipynb)
- [QuantLib-Risks](https://github.com/auto-differentiation/QuantLib-Risks-Py): QuantLib with XAD for fast risk in Python.
- [Python documentation](https://auto-differentiation.github.io/tutorials/python) and the [XAD C++ library](https://github.com/auto-differentiation/xad).